# Target Creation

This notebook creates the target variables for heat risk prediction and splits the data into train/validation/test sets.

## Heat Risk Definition

Heat risk occurs when:
- **(Daily Heat Index >= 105°F) OR (Daily High Temperature >= 95th percentile)** for at least **2 consecutive days**

## Steps:
1. Load processed data
2. TODO: Calculate 95th percentile of daily high temperature
3. TODO: Create binary flags (heat_index_high, temp_percentile_high, heat_risk_day)
4. TODO: Implement rolling window logic for 2 consecutive days
5. TODO: Create final heat_risk target variable
6. TODO: Create regression target (forecast heat index for next day)
7. TODO: Split data temporally (70% train, 15% validation, 15% test)
8. TODO: Save data splits


## Step 1: Mount Google Drive and Import Libraries


In [4]:
# Mount Google Drive and set repo path (Colab)
import os
from pathlib import Path

repo_path = '/content/drive/MyDrive/SHADE-ML-Team'
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    if not os.path.exists(repo_path):
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        # Optionally clone if missing:
        # !git clone https://github.com/HrudithL/SHADE-ML-Team.git /content/drive/MyDrive/SHADE-ML-Team
    os.chdir(repo_path)
    print(f"Using repo at: {os.getcwd()}")
except ImportError:
    # Running locally - set directory to project root
    current_dir = Path(os.getcwd())
    # If we're in a subdirectory (like notebooks), go up to project root
    if current_dir.name == 'notebooks':
        project_root = current_dir.parent
    else:
        # Try to find project root by looking for common markers
        project_root = current_dir
        # Look for .git directory or go up until we find it
        while project_root != project_root.parent:
            if (project_root / '.git').exists() or (project_root / 'README.md').exists():
                break
            project_root = project_root.parent
    
    os.chdir(project_root)
    print(f"Running locally; set CWD to project root: {os.getcwd()}")


Running locally; set CWD to project root: c:\Users\hrudi\OneDrive\Documents\TAMU\SHADE\ML-Repo


In [5]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


Libraries imported successfully!


## Step 2: Load Processed Data


In [7]:
# Load cleaned data from preprocessing step
data_path = 'data/processed/cleaned_data.csv'

if os.path.exists(data_path):
    data = pd.read_csv(data_path)
    print(f"✓ Successfully loaded data from: {data_path}")
    print(f"Shape: {data.shape}")
    print(f"Date range: {data['Date'].min()} to {data['Date'].max()}")
    
    # Ensure Date is datetime
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.sort_values('Date').reset_index(drop=True)
    
    print(f"\nFirst few rows:")
    print(data.head())
else:
    print(f"✗ File not found: {data_path}")
    print("Please run data_preprocessing.ipynb first!")


✓ Successfully loaded data from: data/processed/cleaned_data.csv
Shape: (10354, 27)
Date range: 1999-01-01 to 2023-09-26

First few rows:
        Date  TempHighF  TempLowF  TempAvgF  DewPointAvgF  HumidityAvgPercent  \
0 1999-01-01       73.9      51.8      60.5          57.9                92.0   
1 1999-01-02       64.3      37.2      48.2          30.4                52.1   
2 1999-01-03       39.7      30.3      35.6          12.1                39.0   
3 1999-01-04       42.8      28.0      34.2          13.9                43.7   
4 1999-01-05       55.5      21.4      39.7          31.4                72.6   

   sealevelpressure  VisibilityAvgMiles  PrecipitationSumInches Events  ...  \
0            1008.7                 4.2                     0.0    NaN  ...   
1            1016.6                 9.7                     0.0    NaN  ...   
2            1029.8                 9.9                     0.0    NaN  ...   
3            1034.9                 9.9                    

## Step 3: TODO - Calculate 95th Percentile of Daily High Temperature

**Your task**: Calculate the 95th percentile of daily high temperature.

**Guidelines**:
- Calculate the 95th percentile of `TempHighF` column
- Consider if percentiles should be calculated:
  - Overall (all years combined)
  - Per year (yearly percentiles)
  - Per location (if multiple locations exist)
- Store the percentile value as a threshold

**Code structure**:
```python
# TODO: Calculate 95th percentile
# Option 1: Overall percentile
# temp_95th_percentile = data['TempHighF'].quantile(0.95)

# Option 2: Per year (if needed)
# temp_95th_percentile = data.groupby('year')['TempHighF'].quantile(0.95)

# TODO: Store the percentile value
```


In [8]:
# Try to load 95th percentile from preprocessing
percentile_file = 'data/processed/temp_95th_percentile.txt'

if os.path.exists(percentile_file):
    with open(percentile_file, 'r') as f:
        temp_95th_percentile = float(f.read().strip())
    print(f"✓ Loaded 95th percentile from preprocessing: {temp_95th_percentile:.2f}°F")
else:
    # Calculate if not available
    print("⚠ 95th percentile file not found. Calculating from data...")
    temp_95th_percentile = data['TempHighF'].quantile(0.95)
    print(f"✓ Calculated 95th percentile: {temp_95th_percentile:.2f}°F")

print(f"\n95th percentile threshold: {temp_95th_percentile:.2f}°F")
print(f"This threshold will be used to identify extreme heat days.")


✓ Loaded 95th percentile from preprocessing: 100.30°F

95th percentile threshold: 100.30°F
This threshold will be used to identify extreme heat days.


## Step 4: TODO - Create Binary Flags

**Your task**: Create binary flags indicating when conditions are met.

**Guidelines**:
- `heat_index_high`: 1 if Daily Heat Index >= 105°F, else 0
- `temp_percentile_high`: 1 if Daily High Temperature >= 95th percentile, else 0
- `heat_risk_day`: 1 if day meets either condition (heat_index_high OR temp_percentile_high), else 0

**Code structure**:
```python
# TODO: Create heat_index_high flag
# TODO: Create temp_percentile_high flag
# TODO: Create heat_risk_day flag (OR condition)
```


In [9]:
# Ensure HeatIndex column exists
if 'HeatIndex' not in data.columns:
    print("Calculating HeatIndex column...")
    # Use simplified heat index formula if needed
    def calculate_heat_index(temp_f, humidity):
        T = temp_f
        RH = humidity
        HI = (-42.379 + 
              2.04901523 * T +
              10.14333127 * RH -
              0.22475541 * T * RH -
              6.83783e-3 * T**2 -
              5.481717e-2 * RH**2 +
              1.22874e-3 * T**2 * RH +
              8.5282e-4 * T * RH**2 -
              1.99e-6 * T**2 * RH**2)
        simplified_HI = 0.5 * (T + 61.0 + ((T - 68.0) * 1.2) + (RH * 0.094))
        mask = (T >= 80) & (RH >= 40)
        return np.where(mask, HI, simplified_HI)
    
    data['HeatIndex'] = calculate_heat_index(data['TempAvgF'], data['HumidityAvgPercent'])
    print("✓ HeatIndex calculated")

# Create binary flags
print("\nCreating binary flags...")

# Flag 1: Heat index >= 105°F
data['heat_index_high'] = (data['HeatIndex'] >= 105).astype(int)
print(f"✓ Days with heat index >= 105°F: {data['heat_index_high'].sum()} ({data['heat_index_high'].mean()*100:.2f}%)")

# Flag 2: Temperature >= 95th percentile
data['temp_percentile_high'] = (data['TempHighF'] >= temp_95th_percentile).astype(int)
print(f"✓ Days with temp >= 95th percentile ({temp_95th_percentile:.2f}°F): {data['temp_percentile_high'].sum()} ({data['temp_percentile_high'].mean()*100:.2f}%)")

# Flag 3: Either condition met
data['heat_risk_day'] = ((data['heat_index_high'] == 1) | (data['temp_percentile_high'] == 1)).astype(int)
print(f"✓ Days meeting either condition: {data['heat_risk_day'].sum()} ({data['heat_risk_day'].mean()*100:.2f}%)")



Creating binary flags...
✓ Days with heat index >= 105°F: 4 (0.04%)
✓ Days with temp >= 95th percentile (100.30°F): 519 (5.01%)
✓ Days meeting either condition: 520 (5.02%)


## Step 5: TODO - Implement Rolling Window Logic for 2 Consecutive Days

**Your task**: Identify sequences where heat_risk_day is 1 for at least 2 consecutive days.

**Guidelines**:
- Use rolling window or shift operations to check consecutive days
- A day is considered heat risk if:
  - It meets the condition (heat_index_high OR temp_percentile_high)
  - AND the previous day also meets the condition
- Use pandas shift() or rolling() functions

**Code structure**:
```python
# TODO: Check for 2 consecutive days
# Option 1: Using shift
# data['prev_day_heat_risk'] = data['heat_risk_day'].shift(1)
# data['heat_risk'] = ((data['heat_risk_day'] == 1) & (data['prev_day_heat_risk'] == 1)).astype(int)

# Option 2: Using rolling window
# data['heat_risk'] = (data['heat_risk_day'].rolling(window=2).sum() >= 2).astype(int)
```


In [10]:
# Implement 2-day consecutive logic
print("=" * 80)
print("IMPLEMENTING 2-DAY CONSECUTIVE HEAT RISK LOGIC")
print("=" * 80)

# Method: Use shift to check previous day, and mark current day if either:
# - Current day is heat_risk_day=1 AND previous day is heat_risk_day=1, OR
# - Current day is heat_risk_day=1 AND next day is heat_risk_day=1
# This ensures we mark both days in a 2+ day sequence

data['prev_day_heat_risk'] = data['heat_risk_day'].shift(1).fillna(0)
data['next_day_heat_risk'] = data['heat_risk_day'].shift(-1).fillna(0)

# A day is heat risk if it's part of a 2+ day sequence
data['heat_risk'] = ((data['heat_risk_day'] == 1) & 
                     ((data['prev_day_heat_risk'] == 1) | (data['next_day_heat_risk'] == 1))).astype(int)

# Alternative method using rolling window (commented out):
# data['heat_risk'] = (data['heat_risk_day'].rolling(window=2, min_periods=1).sum() >= 2).astype(int)

print(f"Days with heat_risk_day=1: {data['heat_risk_day'].sum()}")
print(f"Days with heat_risk (2+ consecutive): {data['heat_risk'].sum()}")
print(f"Percentage of heat risk days: {data['heat_risk'].mean()*100:.2f}%")

# Drop temporary columns
data = data.drop(['prev_day_heat_risk', 'next_day_heat_risk'], axis=1)


IMPLEMENTING 2-DAY CONSECUTIVE HEAT RISK LOGIC
Days with heat_risk_day=1: 520
Days with heat_risk (2+ consecutive): 455
Percentage of heat risk days: 4.39%


## Step 6: TODO - Create Regression Target

**Your task**: Create a regression target to forecast heat index for the next day.

**Guidelines**:
- Create `heat_index_next_day`: the heat index value for the next day
- Use shift(-1) to get next day's value
- This will be used for regression tasks (forecasting heat index)
- Handle the last day (no next day) appropriately

**Code structure**:
```python
# TODO: Create regression target
# data['heat_index_next_day'] = data[heat_index_col].shift(-1)
# TODO: Handle NaN values (last day has no next day)
```


In [11]:
# Create regression target: heat index for next day
print("=" * 80)
print("CREATING REGRESSION TARGET")
print("=" * 80)

# Create next day's heat index as regression target
data['heat_index_next_day'] = data['HeatIndex'].shift(-1)

# The last row will have NaN (no next day)
# We'll keep NaN for now - models can handle it or we can drop the last row
n_missing = data['heat_index_next_day'].isnull().sum()
print(f"Regression target created!")
print(f"Missing values in heat_index_next_day: {n_missing} (last day has no next day)")

# Optionally, you can drop the last row or fill with forward fill
# For now, we keep NaN - it will be handled during model training
print(f"✓ Regression target ready for training")
print(f"  Range: {data['heat_index_next_day'].min():.2f}°F to {data['heat_index_next_day'].max():.2f}°F")


CREATING REGRESSION TARGET
Regression target created!
Missing values in heat_index_next_day: 1 (last day has no next day)
✓ Regression target ready for training
  Range: 10.10°F to 118.24°F


## Step 7: TODO - Split Data Temporally

**Your task**: Split data into train (70%), validation (15%), and test (15%) sets while maintaining temporal order.

**Guidelines**:
- **DO NOT shuffle** - maintain chronological order
- Train: first 70% of data
- Validation: next 15% of data
- Test: last 15% of data
- Ensure Date column is sorted before splitting

**Code structure**:
```python
# TODO: Calculate split indices
# TODO: Split data maintaining temporal order
# TODO: Verify splits don't overlap
```


In [13]:
# Split data temporally (maintaining chronological order)
print("=" * 80)
print("TEMPORAL DATA SPLITTING")
print("=" * 80)

# Ensure data is sorted by date
data = data.sort_values('Date').reset_index(drop=True)

# Calculate split indices
n_total = len(data)
train_end_idx = int(n_total * 0.70)
val_end_idx = int(n_total * 0.85)

# Get the date at the split points to ensure we don't split within the same date
train_end_date = data.iloc[train_end_idx]['Date']
val_end_date = data.iloc[val_end_idx]['Date']

# Adjust split indices to ensure all rows with the same date go to the same split
# For train/val split: include all rows with train_end_date in train set
# Find the last index where date equals train_end_date, then add 1 to start validation
train_end_idx = data[data['Date'] == train_end_date].index[-1] + 1
# For val/test split: include all rows with val_end_date in validation set
# Find the last index where date equals val_end_date, then add 1 to start test
val_end_idx = data[data['Date'] == val_end_date].index[-1] + 1

# Split data
train_data = data.iloc[:train_end_idx].copy()
val_data = data.iloc[train_end_idx:val_end_idx].copy()
test_data = data.iloc[val_end_idx:].copy()

print(f"Total samples: {n_total}")
print(f"Train samples: {len(train_data)} ({len(train_data)/n_total*100:.1f}%)")
print(f"Validation samples: {len(val_data)} ({len(val_data)/n_total*100:.1f}%)")
print(f"Test samples: {len(test_data)} ({len(test_data)/n_total*100:.1f}%)")

print(f"\nDate ranges:")
print(f"Train: {train_data['Date'].min()} to {train_data['Date'].max()}")
print(f"Validation: {val_data['Date'].min()} to {val_data['Date'].max()}")
print(f"Test: {test_data['Date'].min()} to {test_data['Date'].max()}")

# Verify no overlap - check that no dates appear in multiple splits
train_dates = set(train_data['Date'].dt.date)
val_dates = set(val_data['Date'].dt.date)
test_dates = set(test_data['Date'].dt.date)

train_val_overlap = train_dates & val_dates
val_test_overlap = val_dates & test_dates

assert len(train_val_overlap) == 0, f"ERROR: Train and validation overlap on dates: {sorted(train_val_overlap)[:5]}"
assert len(val_test_overlap) == 0, f"ERROR: Validation and test overlap on dates: {sorted(val_test_overlap)[:5]}"
print("\n✓ No overlap between splits - temporal order maintained!")


TEMPORAL DATA SPLITTING
Total samples: 10354
Train samples: 7248 (70.0%)
Validation samples: 1553 (15.0%)
Test samples: 1553 (15.0%)

Date ranges:
Train: 1999-01-01 00:00:00 to 2016-05-28 00:00:00
Validation: 2016-05-29 00:00:00 to 2019-06-26 00:00:00
Test: 2019-06-27 00:00:00 to 2023-09-26 00:00:00

✓ No overlap between splits - temporal order maintained!


## Step 8: TODO - Verify Target Variables

**Your task**: Verify that target variables are created correctly.

**Guidelines**:
- Check that heat_risk is binary (0 or 1)
- Verify heat_risk distribution (should be reasonable)
- Check that heat_index_next_day exists and has reasonable values
- Verify no data leakage between splits

**Code structure**:
```python
# TODO: Check target variable distributions
# TODO: Verify data quality
```


In [14]:
# Verify target variables
print("=" * 80)
print("TARGET VARIABLE VERIFICATION")
print("=" * 80)

print("\nHeat Risk Distribution (Overall):")
print(data['heat_risk'].value_counts().sort_index())
print(f"Percentage of heat risk days: {data['heat_risk'].mean()*100:.2f}%")

print("\nHeat Index Next Day Statistics:")
print(data['heat_index_next_day'].describe())

print("\n" + "=" * 80)
print("TARGET VARIABLES IN EACH SPLIT")
print("=" * 80)

for split_name, split_data in [('Train', train_data), ('Validation', val_data), ('Test', test_data)]:
    print(f"\n{split_name}:")
    print(f"  Heat risk: {split_data['heat_risk'].sum()} ({split_data['heat_risk'].mean()*100:.2f}%)")
    print(f"  Heat index next day - Mean: {split_data['heat_index_next_day'].mean():.2f}°F, "
          f"Std: {split_data['heat_index_next_day'].std():.2f}°F")

print("\n✓ Target variables verified!")


TARGET VARIABLE VERIFICATION

Heat Risk Distribution (Overall):
heat_risk
0    9899
1     455
Name: count, dtype: int64
Percentage of heat risk days: 4.39%

Heat Index Next Day Statistics:
count    10353.000000
mean        70.397096
std         16.884124
min         10.104900
25%         57.833400
50%         72.080100
75%         85.004513
max        118.239192
Name: heat_index_next_day, dtype: float64

TARGET VARIABLES IN EACH SPLIT

Train:
  Heat risk: 245 (3.38%)
  Heat index next day - Mean: 69.37°F, Std: 16.61°F

Validation:
  Heat risk: 65 (4.19%)
  Heat index next day - Mean: 73.26°F, Std: 16.94°F

Test:
  Heat risk: 145 (9.34%)
  Heat index next day - Mean: 72.34°F, Std: 17.58°F

✓ Target variables verified!


## Step 9: TODO - Save Data Splits

**Your task**: Save the train, validation, and test splits to CSV files.

**Guidelines**:
- Save to `data/processed/` directory
- Files: `train_data.csv`, `validation_data.csv`, `test_data.csv`
- Include all features and target variables
- Use `index=False` when saving

**Code structure**:
```python
# TODO: Create processed directory if needed
# TODO: Save train_data.csv
# TODO: Save validation_data.csv
# TODO: Save test_data.csv
# TODO: Verify files saved
```


In [15]:
# Save data splits
print("=" * 80)
print("SAVING DATA SPLITS")
print("=" * 80)

# Create directory if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

# Save splits
train_data.to_csv('data/processed/train_data.csv', index=False)
val_data.to_csv('data/processed/validation_data.csv', index=False)
test_data.to_csv('data/processed/test_data.csv', index=False)

print("✓ Data splits saved successfully!")
print("  - train_data.csv")
print("  - validation_data.csv")
print("  - test_data.csv")

# Verify files
for file in ['train_data.csv', 'validation_data.csv', 'test_data.csv']:
    path = f'data/processed/{file}'
    if os.path.exists(path):
        df_check = pd.read_csv(path)
        print(f"  ✓ {file}: {df_check.shape[0]} rows, {df_check.shape[1]} columns")
    else:
        print(f"  ✗ {file}: File not found!")

print("\n✓ All data splits saved and verified!")


SAVING DATA SPLITS
✓ Data splits saved successfully!
  - train_data.csv
  - validation_data.csv
  - test_data.csv
  ✓ train_data.csv: 7248 rows, 32 columns
  ✓ validation_data.csv: 1553 rows, 32 columns
  ✓ test_data.csv: 1553 rows, 32 columns

✓ All data splits saved and verified!


## Summary

After completing all TODO sections, you should have:
- ✅ 95th percentile calculated
- ✅ Binary flags created (heat_index_high, temp_percentile_high, heat_risk_day)
- ✅ 2-day consecutive logic implemented
- ✅ Final heat_risk target variable created
- ✅ Regression target (heat_index_next_day) created
- ✅ Data split temporally (70/15/15)
- ✅ Data splits saved to CSV files

**Next Steps**: Proceed to your team's model notebook:
- LSTM team → `lstm_model.ipynb`
- Random Forest team → `random_forest_model.ipynb`
- XGBoost/AdaBoost team → `xgboost_adaboost_model.ipynb`
